# qGAN Experiment Tutorial

Use this notebook to run or load one selected experiment, run or load a battery, and analyze completed checkpoints.


## 1. Setup

Run this notebook from inside the repository or from `qgan/notebooks`. The cell below makes `qgan/src` importable without installing the package.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
qgan_dir = next(
    (
        candidate
        for root in (cwd, *cwd.parents)
        for candidate in (root, root / "qgan")
        if (candidate / "src" / "qgan_v2").is_dir()
    ),
    None,
)
if qgan_dir is None:
    raise RuntimeError("Could not find the qgan project directory.")

src_path = qgan_dir / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
repo_root = qgan_dir.parent
print({"qgan_dir": qgan_dir, "repo_root": repo_root})

In [ ]:
from qgan_v2.analysis import ThesisReport
from qgan_v2.analysis.report import display_rows
from qgan_v2.experiments import BatteryExperiments, SingleExperiment, manage_runtime_account
from qgan_v2.implementations.registry import IMPLEMENTATIONS
from qgan_v2.visualization import get_visual_config, run_visualization

sorted(IMPLEMENTATIONS)

## 2. IBM Runtime Credentials

This cell is optional. Enable it only when you need to save or verify IBM Runtime credentials.


In [ ]:
SAVE_RUNTIME_ACCOUNT = False
CHECK_RUNTIME_ACCOUNT = False

backend_count = manage_runtime_account(
    save=SAVE_RUNTIME_ACCOUNT,
    check=CHECK_RUNTIME_ACCOUNT,
)
if backend_count is not None:
    print("Available backends:", backend_count)

## 3. Single Experiment

The first YAML file in `configs/singles` is selected by default. Set `single_config_path` to choose another one.


In [ ]:
single_config_path = repo_root / "qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/config.yaml"

single = SingleExperiment.select(
    single_config_path,
    fallback_directory=qgan_dir / "configs" / "singles",
)
single.summary()

### Run Or Load

Choose one boolean. Function arguments stay in the function call.


In [ ]:
RUN_SINGLE_EXPERIMENT = False
state = single.run_or_load(run=RUN_SINGLE_EXPERIMENT, reset_data=False)
print("loaded state:", single.checkpoint)

### Inspect Checkpoint


In [ ]:
print("current_epoch:", state.current_epoch)
print("best_eval:", state.metrics.best_eval())

### Visualize


In [ ]:
VISUALIZE_EXPERIMENT_RESULTS = False

if single.checkpoint.exists() and VISUALIZE_EXPERIMENT_RESULTS:
    run_visualization(
        single.config_path,
        get_visual_config({
            "draw_circuits": False,
            "draw_hardware_layout": True,
            "draw_probs": False,
            "draw_images": True,
            "draw_results": True,
        }),
    )

## 4. Battery Experiments

The first YAML file in `configs/batteries` is selected by default. Set `battery_path` to choose another one.


In [ ]:
battery_path = repo_root / "qgan/configs/batteries/train/train_times_gpu.yaml"

battery = BatteryExperiments.select(
    battery_path,
    fallback_directory=qgan_dir / "configs" / "batteries",
)
print("battery:", battery.battery_path)

### Run Or Load

Set `RUN_BATTERY_EXPERIMENTS = True` to run the battery. Leave it `False` to load existing checkpoints. In both cases, `battery.states` is populated from checkpoint files.


In [ ]:
RUN_BATTERY_EXPERIMENTS = False

battery.run_or_load(
    run=RUN_BATTERY_EXPERIMENTS,
    reset_data=False,
    reset_real_backend_info=False,
    stop_on_error=False,
    overwrite=False,
)

### Battery Summary


In [ ]:
display_rows(battery.config_rows(limit=20))
remaining = len(battery.valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")

## 5. Implementation Adapters

Quick view of implementations in the battery configs loaded above.


In [ ]:
display_rows(
    battery.implementation_rows(project_directory=qgan_dir, limit=20)
)
remaining = len(battery.valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")

## 6. Thesis Results Analysis

The results follow this order: time and feasibility; presets; execution type; gradient method; randomness; scaling; and implementation/packing validation.

Unless stated otherwise, quality figures use completed 1000-epoch simulator runs. CPU/GPU copies of the same scientific run are collapsed for quality analysis, while device remains explicit in timing. Comparison figures show evaluation only, with faint run traces and median/IQR shadows showing variation across seeds. The sole exception is the preset learning-dynamics comparison in Section 2.1, which retains the joined blue generator-loss, red discriminator-loss, and orange evaluation layout.


### Analysis setup and figure export

Set `EXPORT_FIGURES = True` to save PDF and 300-dpi PNG versions under `qgan/figures/thesis_results`. Each subsection keeps its scientific filters and comparison choices visible in the notebook; reusable loading and plotting mechanics live in `qgan_v2.analysis`. KL divergence (`base`) and image-gradient scores (`ang`/`amp`) are faceted rather than pooled because their scales are not comparable.


In [ ]:
EXPORT_FIGURES = False
GPU_MODEL_LABEL = "RTX6000"  # Change if another GPU timing battery is loaded.
PRESET_ORDER = ("base", "ang", "amp")
GRADIENT_ORDER = ("PSR", "REG", "SPSA")

report = ThesisReport.load(
    qgan_dir,
    export_figures=EXPORT_FIGURES,
    gpu_model_label=GPU_MODEL_LABEL,
)
report.results.summary()

### 1. Time, Cost, and Feasibility

Timing comes first so the reader knows which experiments were computationally practical before interpreting model quality.


#### 1.1 Time analysis

These panels use completed five-epoch timing batteries. Median time per epoch is reported because it is less sensitive to a slow initialization epoch. The factors are preset, execution type, gradient method, and `rand0`/`rand1`.

The timing environments are the one-CPU battery (`CPU/1`), the explicit four-CPU battery (`CPU/4`), and the configured GPU battery. Every series is plotted on the exact categorical tick—there is no horizontal series offset.


In [ ]:
TIMING_BASELINE = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "gradient_method": "SPSA",
    "n_qubits": 4,
    "randomness": 0,
}
primary_timing = report.timing_analysis(
    baseline_filters=TIMING_BASELINE,
    factor_levels={"randomness": (0, 1)},
)

#### 1.2 Experimental limitations and feasibility

This subsection records the known resource boundary for each omitted or unfinished experiment class. It deliberately excludes `fake_real`, which is an implementation-validation mode rather than a feasibility result. The completion matrix counts all requests from the current convergence and timing batteries plus the commented noisy-q16 convergence plan. CPU/GPU convergence copies are collapsed into one scientific request, while timing devices remain separate. The adjacent bars partition the matrix's incomplete requests by limitation, so their per-qubit totals match the incomplete counts in the matrix. Files ending in `_old` are excluded throughout.

The categories distinguish time cost, unavailable real-hardware execution time, statevector transpilation limits, and CPU memory limits. They should be interpreted as experimental design constraints, not model-quality outcomes.


In [ ]:
FEASIBILITY_BATTERY_DIR = qgan_dir / "configs" / "batteries" / "train"
INCLUDE_COMMENTED_Q16_REQUESTS = True

feasibility_results = report.feasibility_analysis(
    battery_dir=FEASIBILITY_BATTERY_DIR,
    include_commented_q16=INCLUDE_COMMENTED_Q16_REQUESTS,
)

### 2. Presets

This section establishes how the three data/encoding presets learn and then links the quantitative curves to generated outputs.


#### 2.1 Learning dynamics and best results

The comparison fixes noiseless PSR, `q4`, `rand0`, and seeds 0–2. Each preset has its own column because `base` minimizes KL divergence over a target probability distribution, while `ang` and `amp` use an image-gradient score.

Each preset column retains adversarial losses above evaluation: faint traces identify individual seed runs and solid lines with shadows give the median and IQR. The summary panels show best evaluation and the epoch at which it occurs; point color identifies the seed. This is the only subsection whose learning-dynamics graph includes generator and discriminator losses.


In [ ]:
PRESET_FOCUS = {
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "gradient_method": "PSR",
    "n_qubits": 4,
    "randomness": 0,
}
preset_runs = report.preset_learning_dynamics(
    filters=PRESET_FOCUS,
    presets=PRESET_ORDER,
)

In [ ]:
preset_performance = report.preset_best_results(
    preset_runs,
    presets=PRESET_ORDER,
)

#### 2.2 Initial, last, best, and target outputs

One completed noiseless `q4` run is chosen automatically for each preset using its best evaluation score. If runs tie, gradient methods are preferred in the requested order: PSR, REG, then SPSA. The initial, final-checkpoint, best-checkpoint, and target outputs are rendered together in one `1 × 4` row for each selected run, making the full qualitative change and any regression after the optimum directly visible.


In [ ]:
OUTPUT_FOCUS = {
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "n_qubits": 4,
}
preset_output_manifest = report.select_preset_outputs(
    filters=OUTPUT_FOCUS,
    presets=PRESET_ORDER,
    gradient_priority=GRADIENT_ORDER,
)

In [ ]:
RENDER_PRESET_OUTPUTS = True
QUALITATIVE_RANDOM_SEED = 0

if RENDER_PRESET_OUTPUTS:
    rendered_output_manifest = report.render_preset_outputs(
        random_seed=QUALITATIVE_RANDOM_SEED
    )
else:
    print("Set RENDER_PRESET_OUTPUTS = True to create the four-panel output figures.")

### 3. Execution-Type Comparison

Simulation provides the replicated comparison; the much smaller real-hardware sample is separated so its evidential limits remain explicit.


#### 3.1 Noiseless versus noisy simulation

This comparison uses matched `q4`, SPSA, `rand0`, seeds 0–2. Each preset remains in its own metric column. Learning curves show evaluation only; summaries retain best evaluation and best epoch.


In [ ]:
SIMULATION_EXECUTION_FOCUS = {
    "implementation": "qml_torch",
    "gradient_method": "SPSA",
    "n_qubits": 4,
    "randomness": 0,
    "execution_type": ("noiseless", "noisy"),
}
simulation_execution_runs = report.simulation_execution_dynamics(
    filters=SIMULATION_EXECUTION_FOCUS,
    presets=PRESET_ORDER,
)

In [ ]:
report.simulation_execution_best_results(
    simulation_execution_runs,
    presets=PRESET_ORDER,
)

#### 3.2 Noiseless, noisy, and real hardware

The two available real-QPU runs are compared against matching simulator runs truncated to the same observed epoch budget. This is behavioral evidence for those runs, not a replicated estimate of hardware performance.


In [ ]:
REAL_HARDWARE_FOCUS = {
    "execution_type": "real",
    "implementation": "qml_torch",
    "preset": "base",
    "n_qubits": 4,
    "randomness": 0,
}
hardware_cases = report.real_hardware_comparison(
    filters=REAL_HARDWARE_FOCUS,
    eval_method="kl",
)

### 4. Gradient Method Comparison

SPSA, PSR, and REG are compared with preset, execution type, qubits, randomness, and evaluation metric fixed. Timing is absent here because it is analyzed in Section 1.


In [ ]:
GRADIENT_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "n_qubits": 4,
    "randomness": 0,
    "eval_method": "gradient",
}
gradient_runs = report.gradient_comparison(filters=GRADIENT_FOCUS)

### 5. Randomness Effect

Randomness is first studied as a complete five-level sweep, then with a wider matched `rand0`/`rand1` comparison.


#### 5.1 Complete five-level randomness sweep

The principal analysis requires all five values (`0`, `0.1`, `0.25`, `0.5`, `1`) for seeds 0–2. Evaluation step volatility is the median absolute change between consecutive evaluation scores; larger values indicate a more erratic learning trajectory.


In [ ]:
RANDOMNESS_LEVELS = (0, 0.1, 0.25, 0.5, 1)
RANDOMNESS_SEEDS = (0, 1, 2)
RANDOMNESS_SWEEP_INDEX = 0

randomness_main = report.randomness_sweep(
    levels=RANDOMNESS_LEVELS,
    required_seeds=RANDOMNESS_SEEDS,
    sweep_index=RANDOMNESS_SWEEP_INDEX,
)

#### 5.2 Wider paired rand0/rand1 evidence

The wider dataset is reduced to matched `rand0` and `rand1` pairs. Panels show the change in best evaluation, best epoch, and evaluation step volatility. Zero means no effect; positive score or volatility deltas mean worse or bumpier behavior.


In [ ]:
RANDOMNESS_PAIR_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "eval_method": "gradient",
}
report.randomness_pairs(
    filters=RANDOMNESS_PAIR_FOCUS,
    baseline=0,
    treatment=1,
)

### 6. Scaling Analysis

Scaling is split into preset, execution type, gradient method, and randomness. Each subsection contains representative evaluation dynamics plus best-evaluation and best-epoch scaling across qubit counts. Parameter-count and circuit-width figures are intentionally reserved for the experimental-setup chapter.


#### 6.1 Scaling by preset

Noiseless SPSA and `rand0` are fixed. Each preset is faceted to respect its evaluation metric; qubit count is the compared curve within each facet.


In [ ]:
PRESET_SCALING_FOCUS = {
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "gradient_method": "SPSA",
    "randomness": 0,
}
preset_scaling_runs = report.preset_scaling(
    filters=PRESET_SCALING_FOCUS,
    presets=PRESET_ORDER,
    qubits_by_preset={"amp": (4, 8)},
)

#### 6.2 Scaling by execution type

The dynamics are split into noiseless and noisy panels for `ang`, SPSA, and `rand0`. Within each panel, curves compare the available qubit counts. No noisy `q16` result is inferred.


In [ ]:
EXECUTION_SCALING_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "gradient_method": "SPSA",
    "randomness": 0,
}
EXECUTION_TYPES = ("noiseless", "noisy")
execution_scaling_runs = report.execution_scaling(
    filters=EXECUTION_SCALING_FOCUS,
    execution_types=EXECUTION_TYPES,
)

#### 6.3 Scaling by gradient method

The dynamics are split into PSR, REG, and SPSA panels for `ang`, noiseless execution, and `rand0`. Within each panel, curves compare the available qubit counts.


In [ ]:
GRADIENT_SCALING_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "randomness": 0,
}
gradient_scaling_runs = report.gradient_scaling(
    filters=GRADIENT_SCALING_FOCUS,
    gradient_methods=GRADIENT_ORDER,
)

#### 6.4 Scaling by randomness

The dynamics are split into `rand0` and `rand1` panels for `ang`, noiseless SPSA. Within each panel, curves compare the available qubit counts.


In [ ]:
RANDOMNESS_SCALING_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "gradient_method": "SPSA",
}
RANDOMNESS_SCALING_LEVELS = (0, 1)
randomness_scaling_runs = report.randomness_scaling(
    filters=RANDOMNESS_SCALING_FOCUS,
    randomness_levels=RANDOMNESS_SCALING_LEVELS,
)

### 7. Implementation and Packing Validation

This final section only demonstrates that each implementation path trains. Fake-real `q4`, SPSA, `rand0`, seed-0 traces cover `qml_torch`, `runtime_packed/separate`, and valid `runtime_packed/joined` cases. No deep quality, timing, or statistical claim is made from one seed.


In [ ]:
VALIDATION_FOCUS = {
    "execution_type": "fake_real",
    "n_qubits": 4,
    "gradient_method": "SPSA",
    "randomness": 0,
    "seed": 0,
}
fake_validation = report.implementation_validation(
    filters=VALIDATION_FOCUS,
    presets=PRESET_ORDER,
)